# Lake Erie SCHISM - preparing a mesh for the pipeline

A SCHISM mesh usually arrives with the grid and the physics settings but
without the derived files the pipeline needs. This notebook generates those
with the three `prepare-schism-*` commands, then runs a 12-hour simulation
forced at the open boundary by NOAA's Lake Erie OFS (`leofs`).

The SFINCS counterpart on the same lake and the same window is
[Lake Erie SFINCS](lake_erie_sfincs.ipynb).

## What ships, and what is generated

| Shipped with the mesh | Generated here |
| --- | --- |
| `hgrid.gr3`, `hgrid.ll`, `hgrid.cpp` | `hgrid.nc`, `open_bnds_hgrid.nc` |
| `vgrid.in`, `param.nml`, `bctides.in` | `manning.gr3` |
| `elev.ic`, `windrot_geo2proj.gr3`, `sflux/` | `nwmReaches.csv`, `ngenReaches.csv` |

`param.nml` asks for all three: `nchi = -1` reads `manning.gr3`, and
`if_source = -1` reads the reach crosswalk. Without them SCHISM starts and
runs with no bottom friction file and no river inflow.

## Setup

The mesh, geogrid and hydrofabric are not in the repo. Create the symlinks
once, pointing at wherever yours live:

```bash
cd docs/examples/lake-erie_schism
ln -s /path/to/schism_models/lake_erie        model
ln -s /path/to/geo_em_CONUS.nc                geo_em_CONUS.nc
ln -s /path/to/NWM_v3_hydrofabric.gdb         hydrofabric.gdb
ln -s /path/to/nhf_1.2.2.gpkg                 hydrofabric.gpkg
```

The hydrofabric links point at the files themselves, so it does not matter how
your copies are organised.

In [1]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

notebook_dir = Path.cwd()  # assumes run from docs/examples/notebooks/
example_dir = (notebook_dir.parent / "lake-erie_schism").resolve()
os.chdir(example_dir)

required = (
    "model",
    "geo_em_CONUS.nc",
    "hydrofabric.gdb",
    "hydrofabric.gpkg",
    "run.yaml",
)
missing = [n for n in required if not Path(n).exists()]
if missing:
    raise FileNotFoundError(f"Missing in {example_dir}: {missing}. See ../README.md.")


def cli(*args: str) -> None:
    """Run a coastal-calibration CLI command and stream its output."""
    print(f"$ coastal-calibration {' '.join(args)}\n")
    subprocess.run([sys.executable, "-m", "coastal_calibration.cli", *args], check=True)


print(f"Working directory: {example_dir}")
print("Mesh contents:", sorted(p.name for p in Path("model").iterdir()))

Working directory: docs/examples/lake-erie_schism
Mesh contents: ['.esa_worldcover_cache', 'bctides.in', 'elev.ic', 'hgrid.cpp', 'hgrid.gr3', 'hgrid.ll', 'param.nml', 'sflux', 'vgrid.in', 'windrot_geo2proj.gr3']


The mesh boundary can be pulled out in QGIS with the `nwm_coastal` plugin,
which is how the domain outline below was made. It is only for looking at -
the prepare commands read `hgrid.gr3` directly.

![Lake Erie SCHISM mesh boundary extracted in QGIS](../images/Lake_Erie_SCHISM_extract_mesh_boundary.jpg)

## 1. ESMF mesh files

Converts `hgrid.gr3` into the NetCDF form the atmospheric regridder and the
boundary stages read.

In [2]:
cli("prepare-schism-mesh", "./model")

$ coastal-calibration prepare-schism-mesh ./model



Writing ESMF NetCDF files from hgrid.gr3                                        



Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/lake_erie/hgrid.nc
         /home/laurscham/ngwpc/run_coastal/schism_models/lake_erie/open_bnds_hgrid.nc

  nodes:                 975,042
  elements:              1,926,950
  max nodes per element: 3
  open boundary nodes:   4,415


Open boundaries: 4,415 nodes in 2 segments                                      
ESMF mesh written: 975,042 nodes, 1,926,950 elements, max 3 nodes/element       


## 2. Manning roughness

Samples ESA WorldCover land cover at every node and maps each class to a
Manning's n. Add `--force` to overwrite an existing file.

In [3]:
if Path("model/manning.gr3").exists():
    print("manning.gr3 already present; pass --force to regenerate")
else:
    cli("prepare-schism-manning", "./model")

$ coastal-calibration prepare-schism-manning ./model



ESA WorldCover: all 4 tile(s) already cached                                    



Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/lake_erie/manning.gr3

  nodes:            975,042
  elements:         1,926,950
  WorldCover tiles: 4 (0 missing)
  fallback nodes:   0 (0.00%)
  manning n:        min 0.020  mean 0.063  max 0.120
  distinct values:  8


## 3. Reach crosswalks

Intersects the mesh boundary with a hydrofabric and writes one row per
crossing: a flowline entering the mesh becomes a source, one leaving it
becomes a sink. `nwmReaches.csv` keys on NWM COMIDs and is what `nwm_ana`
and `nwm_retro` runs use; `ngenReaches.csv` keys on NextGen `fp_id` for
`ngen_forecast` runs.

In [4]:
cli(
    "prepare-schism-reaches",
    "./model",
    "--nwm-gdb",
    "./hydrofabric.gdb",
    "--ngen-gpkg",
    "./hydrofabric.gpkg",
    "--force",
)

$ coastal-calibration prepare-schism-reaches ./model --nwm-gdb ./hydrofabric.gdb --ngen-gpkg ./hydrofabric.gpkg --force



Inferred hydrofabric region: conus


nwm_reaches_conus: 24636 flowline(s) in the mesh bounding box                   


nwm_reaches_conus: 535 flowline(s) cross a land boundary                        


nwm_reaches_conus: 553 inbound and 56 outbound crossing(s)                      


...

flowpaths: 425 flowline(s) cross a land boundary                                
flowpaths: 427 inbound and 40 outbound crossing(s)                              


testing 184290 candidate element(s) near the boundary                           


resolved 467 point(s) inside an element, 0 snapped to the nearest               



Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/lake_erie/nwmReaches.csv
  layer:                 nwm_reaches_conus
  flowlines in bbox:     24,636
  crossing the boundary: 535
  inbound crossings:     553
  outbound crossings:    56
  merged into an element:0
  resolved by element:   609
  resolved by nearest:   0
  source rows:           553 (521 distinct reaches)
  sink rows:             56 (43 distinct reaches)

Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/lake_erie/ngenReaches.csv
  layer:                 flowpaths
  flowlines in bbox:     12,137
  crossing the boundary: 425
  inbound crossings:     427
  outbound crossings:    40
  merged into an element:0
  resolved by element:   467
  resolved by nearest:   0
  source rows:           427 (412 distinct reaches)
  sink rows:             40 (36 distinct reaches)


wrote 427 source(s) and 40 sink(s) to                                           
/home/laurscham/ngwpc/run_coastal/schism_models/lake_erie/ngenReaches.csv       


### What landed

The source block should be dominated by the Detroit River, and the sink block
by the Niagara - Erie's inflow and outflow. If the largest reach shows up as a
source rather than a sink, the crossing direction was read wrong.

In [5]:
from coastal_calibration.schism.ngen_reaches import read_reaches_blocks

for name in ("nwmReaches.csv", "ngenReaches.csv"):
    sources, sinks = read_reaches_blocks(Path("model") / name)
    print(f"{name}: {len(sources)} sources, {len(sinks)} sinks")

for name in ("hgrid.nc", "open_bnds_hgrid.nc", "manning.gr3"):
    p = Path("model") / name
    print(f"{name}: {p.stat().st_size / 1e6:.1f} MB")

nwmReaches.csv: 553 sources, 56 sinks
ngenReaches.csv: 427 sources, 40 sinks
hgrid.nc: 71.5 MB
open_bnds_hgrid.nc: 71.5 MB
manning.gr3: 91.0 MB


## 4. The run configuration

Everything above is preparation and only has to happen once per mesh. The
cell after this one is the simulation, which is the slow part - stop here if
you only wanted the model built.

In [6]:
print(Path("run.yaml").read_text())

# Lake Erie (LEOFS) - SCHISM run configuration
#
# The mesh is reached through the ./model symlink; see ../README.md. Every
# file SCHISM needs beyond the shipped mesh is generated by the three
# prepare-schism-* commands in ../notebooks/lake_erie_schism.ipynb.
#
# Usage:
#   coastal-calibration run run.yaml --dry-run
#   coastal-calibration run run.yaml

model: schism

simulation:
  start_date: 2026-08-15 00:00:00
  duration_hours: 12
  coastal_domain: greatlakes  # required by boundary.source glofs, and vice versa
  meteo_source: nwm_ana       # CONUS NWM forcing covers the lakes

boundary:
  source: glofs               # NOAA Great Lakes OFS (FVCOM) nowcast water levels
  glofs_model: leofs          # Lake Erie OFS

paths:
  work_dir: ./run
  raw_download_dir: ./downloads

download:
  enabled: true

model_config:
  prebuilt_dir: ./model
  geogrid_file: ./geo_em_CONUS.nc
  # GLOFS is on the lake's low water datum; this mesh carries absolute IGLD85
  # elevations (elev.ic = 174.73). 1

Two settings are specific to a Great Lakes mesh:

- `forcing_to_mesh_offset_m: 173.4` - GLOFS reports water levels on the
  lake's low water datum while this mesh carries absolute IGLD85 elevations
  (`elev.ic` starts at 174.73), so the forcing is shifted up by 173.5, less
  0.1 for a LEOFS high bias of about that size. Gauge observations are
  separate: they come in IGLD and are shifted by `vdatum_mesh_to_msl_m`,
  which is `0.0` because this mesh is already IGLD.
- `coastal_domain: greatlakes` and `boundary.source: glofs` are required
  together; either alone fails validation.

`ntasks_per_node: 16` suits a workstation. Each rank holds roughly 1.5 GB on
this 1.9M-element mesh, so memory runs out before cores do.

In [7]:
cli("run", "run.yaml", "--dry-run")

$ coastal-calibration run run.yaml --dry-run



Dry run mode - validating configuration...                                      
Dry run mode - validation passed, no execution                                  
Workflow completed successfully.                                                


## 5. Run it

12 hours of simulation. The solver is the bulk of the wall time; everything
before it - forcing, sflux, boundary, discharge, partitioning - takes a
couple of minutes.

In [8]:
cli("run", "run.yaml")

$ coastal-calibration run run.yaml



Coastal Calibration Workflow                                                    
Start Time: 2026-09-28 03:40:13                                                 
----------------------------------------                                        
Cleaned generated files from                                                    
docs/examples/lake-erie_schism/run            
Stage: download                                                                 
Start Time: 2026-09-28 03:40:13                                                 
  Download input data (NWM, GLOFS)                                              


GLOFS leofs: fetching 13 hour(s) from NCEI                                      


GLOFS leofs: 13/13 hour(s) fetched                                              
  meteo/nwm_ana: 13/13 [OK]                                                     
  hydro/nwm: 13/13 [OK]                                                         
  coastal/glofs: 13/13 [OK]                                                     
  Total: 39/39 (failed: 0)                                                      
GLOFS leofs: merged 13 hour(s) into glofs_leofs_2026081500_2026081512.nc        
  Download complete. Raw files stored in                                        
docs/examples/lake-erie_schism/downloads      
  [✓] COMPLETED (3m 47s)                                                        
----------------------------------------                                        
Stage: schism_forcing_prep                                                      
Start Time: 2026-09-28 03:44:00                                                 
  Prepare LDASIN forcing data                                 

  NWM forcing generated in                                                      
docs/examples/lake-erie_schism/run/forcing_inp
ut                                                                              
  [✓] COMPLETED (1m 3s)                                                         
----------------------------------------                                        
Stage: schism_sflux                                                             
Start Time: 2026-09-28 03:45:04                                                 
  Generate sflux atmospheric files                                              
  Generating sflux from LDASIN files                                            
    No hgrid.ll or hgrid.gr3 in                                                 
docs/examples/lake-erie_schism/run; writing   
full geogrid (no subsetting).                                                   


...

    Partitioned mesh into 11 compute ranks →                                    
docs/examples/lake-erie_schism/run/partition.p
rop                                                                             
  Set iout_sta = 1, nspool_sta = 18 in param.nml                                
  SCHISM pre-processing complete                                                
  [✓] COMPLETED (16s)                                                           
----------------------------------------                                        
Stage: schism_run                                                               
Start Time: 2026-09-28 03:46:33                                                 
  Run SCHISM model (MPI)                                                        
  Launching pschism with 16 MPI tasks (1 node(s), 5 scribe(s))                  
  Command: mpiexec -n 16                                                        
.pixi/envs/dev/bin/pschism 5                  


  SCHISM run completed successfully (output:                                    
docs/examples/lake-erie_schism/run/outputs/sch
ism_log.txt)                                                                    
  [✓] COMPLETED (1h 20m 53s)                                                    
----------------------------------------                                        
Stage: schism_postprocess                                                       
Start Time: 2026-09-28 05:07:26                                                 
  Post-process SCHISM outputs                                                   
    No hotstart_000000_*.nc files found in                                      
docs/examples/lake-erie_schism/run/outputs;   
nothing to combine                                                              
  SCHISM post-processing complete                                               
  [✓] COMPLETED (0s)                                                            
---------------

  Successfully retrieved data for 11 station(s)                                 


  Saved 3 comparison figure(s) to                                               
docs/examples/lake-erie_schism/run/figs       
  Including 11 NOAA gauge(s) from station.in                                    


  Wrote obs_water_level.parquet with 11 station(s) and 12 timestep(s)           
  [✓] COMPLETED (10s)                                                           
----------------------------------------                                        
Workflow COMPLETED | Total Duration: 1:27:23                                    
                                                                                
Timing Summary:                                                                 
----------------------------------------                                        
  [✓] download: 3m 47s                                                          
  [✓] schism_forcing_prep: 0s                                                   
  [✓] schism_forcing: 1m 3s                                                     
  [✓] schism_sflux: 38s                                                         
  [✓] schism_params: 0s                                                         
  [✓] schism_obs: 4s        

## Results

`run/figs/` holds the water-level comparison against the NOAA CO-OPS gauges
inside the domain, and `run/outputs/` the SCHISM NetCDF output.

In [9]:
figs = sorted(Path("run/figs").glob("*.png")) if Path("run/figs").exists() else []
print(f"figures: {[p.name for p in figs]}")

outputs = sorted(Path("run/outputs").glob("out2d_*.nc")) if Path("run/outputs").exists() else []
print(f"outputs: {[p.name for p in outputs]}")

figures: ['stations_comparison_001.png', 'stations_comparison_002.png', 'stations_comparison_003.png']
outputs: ['out2d_1.nc', 'out2d_10.nc', 'out2d_11.nc', 'out2d_12.nc', 'out2d_2.nc', 'out2d_3.nc', 'out2d_4.nc', 'out2d_5.nc', 'out2d_6.nc', 'out2d_7.nc', 'out2d_8.nc', 'out2d_9.nc']


## Summary

Three commands turned a shipped mesh into a runnable model:
`prepare-schism-mesh`, `prepare-schism-manning`, `prepare-schism-reaches`.
They are per-mesh, not per-run, so a second simulation on this domain only
needs a new `run.yaml`.

To model a different lake, point `model` at that mesh, set `glofs_model` to
its OFS (`lmhofs`, `loofs`, `lsofs`) and check the mesh datum before trusting
`forcing_to_mesh_offset_m`.